# Week 6: Logistic Regression

Goal: predict the game winner using matchup stats.
`y = 1` means team1 wins, `y = 0` means team2 wins.

Train on seasons <= 2021, validate on 2022, test on 2023-2025.
Metrics: accuracy, precision, recall, and confusion matrix.



In [33]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import ParameterGrid
from sklearn.metrics import accuracy_score, precision_score, recall_score, confusion_matrix
from IPython.display import display
import pickle



In [ ]:
#load data
projectDir = Path.cwd().resolve().parent
dataPath = projectDir / "data" / "processed" / "matchupDiff_week5_features.csv"

matchupDiff = pd.read_csv(dataPath)
matchupDiff["y"] = matchupDiff["y"].astype(int)



In [ ]:
#dropping any columns that could cause data leakage
leakCols = [
    "WScore", "LScore", "WTeamID", "LTeamID", "winnerTeamId",
    "team1Name", "team2Name", "Unnamed: 0"
]
idCols = ["team1Id", "team2Id"]

#all dropped columns
dropCols = ["Season", "y"] + leakCols + idCols



In [ ]:
# train = 2003 - 2021, validation 2022, test 2023 - 2025
trainDf = matchupDiff[matchupDiff["Season"] <= 2021].copy()
valDf = matchupDiff[matchupDiff["Season"] == 2022].copy()
testDf = matchupDiff[matchupDiff["Season"].between(2023, 2025)].copy()



In [ ]:
def buildXy(splitDf, featureCols=None): #drops columns, only contains numeric features reindexes columns, and returns x,y, and feature columns
    x = splitDf.drop(columns=dropCols, errors="ignore") #dropping columns ignoring error messages
    x = x.select_dtypes(include=[np.number])

    if featureCols is None:
        featureCols = x.columns.tolist()
    else:
        x = x.reindex(columns=featureCols)

    y = splitDf["y"].astype(int)
    return x, y, featureCols

xTrain, yTrain, featureCols = buildXy(trainDf)
xVal, yVal, _ = buildXy(valDf, featureCols)
xTest, yTest, _ = buildXy(testDf, featureCols)

print(f"Train shape: {xTrain.shape}, Val shape: {xVal.shape}, Test shape: {xTest.shape}")



Train shape: (1162, 108), Val shape: (63, 108), Test shape: (128, 108)


In [38]:
def evaluateModel(pipeline, x, y, threshold=0.5):
    proba = pipeline.predict_proba(x)[:, 1]
    preds = (proba >= threshold).astype(int)

    metrics = {
        "accuracy": accuracy_score(y, preds),
        "precision": precision_score(y, preds, zero_division=0),
        "recall": recall_score(y, preds, zero_division=0)
    }

    cm = confusion_matrix(y, preds)
    cmDf = pd.DataFrame(cm, index=["Actual 0", "Actual 1"], columns=["Pred 0", "Pred 1"])
    return metrics, cmDf



In [ ]:
#parameter grid 
paramGrid = {
    "cValue": [0.1, 0.3, 0.6, 0.9, 1.0, 1.5],
    "penalty": ["l1", "l2"],
    "classWeight": [None, "balanced", {0: 1, 1: 1.5}, {0: 1, 1: 2}],
    "threshold": [0.35, 0.4, 0.45, 0.5, 0.55, 0.6, 0.65],
    "maxIter": [5000],
    "tol": [1e-4, 1e-3],
    "fitIntercept": [True, False]
}

configs = list(ParameterGrid(paramGrid))
print(f"Total model configs: {len(configs)}")


Total model configs: 1344


In [40]:
results = []

for idx, cfg in enumerate(configs):
    pipe = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(
            solver="saga",
            penalty=cfg["penalty"],
            C=cfg["cValue"],
            class_weight=cfg["classWeight"],
            max_iter=cfg["maxIter"],
            tol=cfg["tol"],
            fit_intercept=cfg["fitIntercept"],
            random_state=226,
            n_jobs=-1
        ))
    ])

    pipe.fit(xTrain, yTrain)

    trainMetrics, _ = evaluateModel(pipe, xTrain, yTrain, cfg["threshold"])
    valMetrics, _ = evaluateModel(pipe, xVal, yVal, cfg["threshold"])

    results.append({
        "configId": idx,
        "model": str(cfg),
        "set": "train",
        **trainMetrics
    })
    results.append({
        "configId": idx,
        "model": str(cfg),
        "set": "val",
        **valMetrics
    })

resultsDf = pd.DataFrame(results)
resultsDf


,configId,model,set,accuracy,precision,recall
0,0,"{'cValue': 0.1, 'classWeight': None, 'fitInter...",train,0.674699,0.674699,1.000000
1,0,"{'cValue': 0.1, 'classWeight': None, 'fitInter...",val,0.634921,0.634921,1.000000
2,1,"{'cValue': 0.1, 'classWeight': None, 'fitInter...",train,0.674699,0.674699,1.000000
3,1,"{'cValue': 0.1, 'classWeight': None, 'fitInter...",val,0.634921,0.634921,1.000000
4,2,"{'cValue': 0.1, 'classWeight': None, 'fitInter...",train,0.674699,0.674699,1.000000
...,...,...,...,...,...,...
2683,1341,"{'cValue': 1.5, 'classWeight': {0: 1, 1: 2}, '...",val,0.476190,0.818182,0.225000
2684,1342,"{'cValue': 1.5, 'classWeight': {0: 1, 1: 2}, '...",train,0.379518,0.811881,0.104592
2685,1342,"{'cValue': 1.5, 'classWeight': {0: 1, 1: 2}, '...",val,0.444444,1.000000,0.125000
2686,1343,"{'cValue': 1.5, 'classWeight': {0: 1, 1: 2}, '...",train,0.379518,0.811881,0.104592


In [41]:
# Pick the winner on validation
valOnly = resultsDf[resultsDf["set"] == "val"].copy()
valOnly = valOnly.sort_values(["accuracy", "recall", "precision"], ascending=False)

bestRow = valOnly.iloc[0]
bestCfg = configs[int(bestRow["configId"])]

print("Best model (by validation accuracy, then recall, precision):")
print(bestRow)


Best model (by validation accuracy, then recall, precision):
configId                                                  1318
model        {'cValue': 1.5, 'classWeight': {0: 1, 1: 2}, '...
set                                                        val
accuracy                                              0.698413
precision                                             0.690909
recall                                                    0.95
Name: 2637, dtype: object


In [ ]:
#pipeline to find best parameters based on accuracy https://scikit-learn.org/stable/model_selection.html
bestPipe = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(
        solver="saga",
        penalty=bestCfg["penalty"],
        C=bestCfg["cValue"],
        class_weight=bestCfg["classWeight"],
        max_iter=bestCfg["maxIter"],
        tol=bestCfg["tol"],
        fit_intercept=bestCfg["fitIntercept"],
        random_state=226,
        n_jobs=-1
    ))
])

bestPipe.fit(xTrain, yTrain)

trainMetrics, trainCm = evaluateModel(bestPipe, xTrain, yTrain, bestCfg["threshold"])
valMetrics, valCm = evaluateModel(bestPipe, xVal, yVal, bestCfg["threshold"])

print("Train metrics:", trainMetrics)
display(trainCm)

print("Validation metrics:", valMetrics)
display(valCm)



Train metrics: {'accuracy': 0.6867469879518072, 'precision': 0.7134146341463414, 'recall': 0.8954081632653061}


,Pred 0,Pred 1
Actual 0,96,282
Actual 1,82,702


Validation metrics: {'accuracy': 0.6984126984126984, 'precision': 0.6909090909090909, 'recall': 0.95}


,Pred 0,Pred 1
Actual 0,6,17
Actual 1,2,38


In [ ]:
#use best model on the test 
trainValDf = matchupDiff[matchupDiff["Season"] <= 2022].copy()
xTrainVal, yTrainVal, _ = buildXy(trainValDf, featureCols)

bestPipe.fit(xTrainVal, yTrainVal)

testMetrics, testCm = evaluateModel(bestPipe, xTest, yTest, bestCfg["threshold"])
print("Test metrics (2023-2025):", testMetrics)
display(testCm)



Test metrics (2023-2025): {'accuracy': 0.640625, 'precision': 0.6788990825688074, 'recall': 0.8705882352941177}


,Pred 0,Pred 1
Actual 0,8,35
Actual 1,11,74


In [45]:
#exporting model to pickle
modelDir = projectDir / "models"
modelDir.mkdir(parents=True, exist_ok=True)

modelPath = modelDir / "logreg_week6_best.pkl"
payload = {
    "model": bestPipe,
    "threshold": bestCfg["threshold"],
    "featureCols": featureCols,
    "params": bestCfg
}

with open(modelPath, "wb") as f:
    pickle.dump(payload, f)



In [51]:
modelDir = projectDir / "models"
modelDir.mkdir(parents=True, exist_ok=True)

modelPath = modelDir / "logreg_week6_best.pkl"
with open(modelPath, "wb") as f:
    pickle.dump(payload, f)
